[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/breakpoints.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239469-lesson-2-breakpoints)

# 断点

## 回顾

对于`人工参与循环`，我们通常想要在图运行时看到其输出。

我们通过流式传输为此奠定了基础。

## 目标

现在，让我们讨论`人工参与循环`的动机：

(1) `批准` - 我们可以中断我们的agent，向用户显示状态，并允许用户接受某个行动

(2) `调试` - 我们可以回退图以重现或避免问题

(3) `编辑` - 你可以修改状态

LangGraph提供了几种方式来获取或更新agent状态，以支持各种`人工参与循环`工作流。

首先，我们将介绍[断点](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/breakpoints/#simple-usage)，它提供了一种在特定步骤停止图的简单方法。

我们将展示这如何实现用户`批准`。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_openai langgraph_sdk langgraph-prebuilt

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

## 用于人工批准的断点

让我们重新考虑在模块1中使用的简单agent。

让我们假设我们担心工具使用：我们想要批准agent使用其任何工具。

我们只需要简单地使用`interrupt_before=["tools"]`编译图，其中`tools`是我们的工具节点。

这意味着执行将在节点`tools`之前被中断，该节点执行工具调用。

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """将a和b相乘。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将是一个工具
def add(a: int, b: int) -> int:
    """将a和b相加。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """将a除以b。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="您是一个有用的助手，负责对一组输入执行算术运算。")

# 节点
def assistant(state: MessagesState):
   """助手节点，处理消息并调用LLM"""
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些做实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些确定控制流
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自assistant的最新消息（结果）是工具调用 -> tools_condition路由到tools
    # 如果来自assistant的最新消息（结果）不是工具调用 -> tools_condition路由到END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(interrupt_before=["tools"], checkpointer=memory)

# 显示
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# 输入
initial_input = {"messages": HumanMessage(content="计算2乘以3")}

# 线程
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

我们可以获取状态并查看要调用的下一个节点。

这是查看图已被中断的好方法。

In [ ]:
state = graph.get_state(thread)
state.next

现在，我们将介绍一个巧妙的技巧。

当我们使用`None`调用图时，它将从最后的状态检查点继续！

![breakpoints.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbae7985b747dfed67775d_breakpoints1.png)

为了清晰起见，LangGraph将重新发出当前状态，其中包含带有工具调用的`AIMessage`。

然后它将继续执行图中的后续步骤，从工具节点开始。

我们看到工具节点使用此工具调用运行，并且它被传递回聊天模型以获得我们的最终答案。

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，让我们将这些与接受用户输入的特定用户批准步骤结合起来。

In [ ]:
# 输入
initial_input = {"messages": HumanMessage(content="计算2乘以3")}

# 线程
thread = {"configurable": {"thread_id": "2"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

# 获取用户反馈
user_approval = input("您想要调用工具吗？(yes/no): ")

# 检查批准
if user_approval.lower() == "yes":
    
    # 如果批准，继续图执行
    for event in graph.stream(None, thread, stream_mode="values"):
        event['messages'][-1].pretty_print()
        
else:
    print("用户取消了操作。")

### 与LangGraph API一起使用断点

**⚠️ 免责声明**

自从拍摄这些视频以来，我们更新了Studio，使其可以在本地运行并在浏览器中打开。这现在是运行Studio的首选方式（而不是像视频中显示的那样使用桌面应用程序）。请参阅[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)的本地开发服务器文档和[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)的相关说明。要启动本地开发服务器，请在此模块的`/studio`目录中的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

LangGraph API [支持断点](https://langchain-ai.github.io/langgraph/cloud/how-tos/human_in_the_loop_breakpoint/#sdk-initialization)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("很抱歉，Google Colab目前不支持LangGraph Studio")

In [ ]:
# 这是本地开发服务器的URL
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

如上所示，我们可以在编译在Studio中运行的图时添加`interrupt_before=["node"]`。

但是，使用API，你也可以直接将`interrupt_before`传递给stream方法。

In [ ]:
initial_input = {"messages": HumanMessage(content="计算2乘以3")}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="values",
    interrupt_before=["tools"],
):
    print(f"接收到类型为: {chunk.event}的新事件...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)

现在，我们可以通过传递`thread_id`和`None`作为输入，像之前一样从断点继续！

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    "agent",
    input=None,
    stream_mode="values",
    interrupt_before=["tools"],
):
    print(f"接收到类型为: {chunk.event}的新事件...")
    messages = chunk.data.get('messages', [])
    if messages:
        print(messages[-1])
    print("-" * 50)